# 02 - Feature Engineering

Builds on the findings from `01_eda.ipynb`. This notebook:

1. Cleans the known data-quality issues (sign-flipped `weight`, missing `weight`/`market_index`).
2. Engineers distance, temporal, market-signal, and categorical features.
3. Handles the two structural challenges identified in the EDA:
   - **Unseen cities** in `validation.csv` (8 cities never seen in training) — handled with a smoothed target encoder that falls back to the global mean for anything unseen.
   - **`december-chart-inputs.csv` has no `market_index`/`quote_signal`** — handled by building a reduced feature schema that never depends on those two columns, usable for both December and a fallback model trained without them.
4. Produces a **time-based train/holdout split** (not random) since `validation.csv` and the December file both fall after the end of the training window, so a random split would overstate real-world performance.
5. Saves processed feature tables and encoder artifacts to `data/processed/` for the modeling notebook to consume directly.

## 1. Setup & Imports

In [1]:
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

## 2. Load Raw Data

In [2]:
TRAIN_PATH = "../data/train-test.csv"
VALID_PATH = "../data/validation.csv"
DECEMBER_PATH = "../data/december-chart-inputs.csv"

train_raw = pd.read_csv(TRAIN_PATH)
validation_raw = pd.read_csv(VALID_PATH)
december_raw = pd.read_csv(DECEMBER_PATH)

train_raw["date"] = pd.to_datetime(train_raw["date"])
validation_raw["date"] = pd.to_datetime(validation_raw["date"])
december_raw["date"] = pd.to_datetime(december_raw["date"])

print("train:", train_raw.shape)
print("validation:", validation_raw.shape)
print("december:", december_raw.shape)

train: (48000, 14)
validation: (12000, 13)
december: (31, 7)


## 3. Cleaning Functions

Two issues were identified in the EDA, both handled here as small, reusable functions so the same logic can be applied consistently to train, validation, and December.

- **Negative `weight` values** (sign-flip error): corrected with `abs(weight)`.
- **Missing `weight` / `market_index`**: a binary "was missing" indicator is added *before* imputation (in case missingness itself is informative), then values are filled with the training median.

In [3]:
def clean_weight(df: pd.DataFrame) -> pd.DataFrame:
    """Corrects the sign-flip data-entry error identified in the EDA."""
    df = df.copy()
    df["weight"] = df["weight"].abs()
    return df


def add_missing_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Flags rows that were missing weight/market_index BEFORE imputation."""
    df = df.copy()
    df["weight_missing"] = df["weight"].isna().astype(int)
    if "market_index" in df.columns:
        df["market_index_missing"] = df["market_index"].isna().astype(int)
    return df


def impute_numeric(df: pd.DataFrame, fill_values: dict) -> pd.DataFrame:
    """Fills missing numeric values using externally-supplied (train-derived) statistics."""
    df = df.copy()
    for col, value in fill_values.items():
        if col in df.columns:
            df[col] = df[col].fillna(value)
    return df

## 4. Rate-per-Mile Outlier Flag

671 training rows (1.40%) had a rate-per-mile far outside the tight band the rest of the data follows, independent of the weight sign-flip issue. Rather than silently dropping them, they are **flagged** so the modeling notebook can experiment with including vs. excluding them and measure the effect on holdout error. This can only be computed on `train-test.csv`, since it requires the target.

In [4]:
RPM_LOWER, RPM_UPPER = 1.0, 3.5

def flag_rpm_outliers(df: pd.DataFrame, lower: float = RPM_LOWER, upper: float = RPM_UPPER) -> pd.DataFrame:
    """Only valid on labeled data (requires posted_rate)."""
    df = df.copy()
    rate_per_mile = df["posted_rate"] / df["distance"]
    df["is_rpm_outlier"] = ((rate_per_mile < lower) | (rate_per_mile > upper)).astype(int)
    return df

train_flagged = flag_rpm_outliers(train_raw)
print(f"Flagged {train_flagged['is_rpm_outlier'].sum()} rows "
      f"({train_flagged['is_rpm_outlier'].mean()*100:.2f}%) as rate-per-mile outliers")

Flagged 671 rows (1.40%) as rate-per-mile outliers


## 5. Distance Features

The EDA showed rate-per-mile declines non-linearly with distance. A log transform gives linear models a chance to capture that curve; tree-based models can use raw `distance` directly, so both are kept.

In [5]:
def add_distance_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["log_distance"] = np.log1p(df["distance"])
    return df

## 6. Temporal Features

`validation.csv` (November) and `december-chart-inputs.csv` (December) both fall **after** the last training date (October 31), so the model must extrapolate the seasonal cycle found in the EDA rather than interpolate within it.

- `month_sin` / `month_cos`: a smooth, cyclical encoding of day-of-year. Unlike a raw month number or one-hot month, a cyclical encoding extrapolates sensibly into November/December because it's built from a continuous sine/cosine wave rather than categories the model has never seen.
- `dow_sin` / `dow_cos`: cyclical day-of-week (minor effect per the EDA, included for completeness).
- `days_since_start`: a linear day-count from the first training date, to let the model pick up any longer-term trend on top of the seasonal cycle.

In [6]:
def add_temporal_features(df: pd.DataFrame, reference_date: pd.Timestamp) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    day_of_year = df["date"].dt.dayofyear
    df["month_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
    df["month_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)

    day_of_week = df["date"].dt.dayofweek
    df["dow_sin"] = np.sin(2 * np.pi * day_of_week / 7)
    df["dow_cos"] = np.cos(2 * np.pi * day_of_week / 7)

    df["days_since_start"] = (df["date"] - reference_date).dt.days
    return df

REFERENCE_DATE = train_raw["date"].min()
print("Reference date (day 0):", REFERENCE_DATE.date())

Reference date (day 0): 2025-01-01


## 7. Market Signal Interaction Features

Section 12 of the EDA showed `market_index` and `quote_signal` correlate weakly with `posted_rate` on their own (~0.03 to -0.04) but strongly once multiplied by `distance` (0.88-0.90) - they scale the base distance-driven rate rather than shifting it additively. These interaction terms are added wherever the two source columns are present (they never are in the December file, so this function is a no-op there).

In [7]:
def add_market_interactions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "market_index" in df.columns:
        df["market_index_x_distance"] = df["market_index"] * df["distance"]
    if "quote_signal" in df.columns:
        df["quote_signal_x_distance"] = df["quote_signal"] * df["distance"]
    return df

## 8. Equipment Encoding

`equipment` has only 3 low-cardinality classes (Dry Van, Reefer, Flatbed) present identically in every file, so a plain one-hot encoding is safe - there's no unseen-category risk here (unlike `pickup`/`delivery`).

In [8]:
EQUIPMENT_CATEGORIES = ("Dry Van", "Reefer", "Flatbed")

def one_hot_equipment(df: pd.DataFrame, categories=EQUIPMENT_CATEGORIES) -> pd.DataFrame:
    df = df.copy()
    for category in categories:
        column_name = f"equip_{category.replace(' ', '_')}"
        df[column_name] = (df["equipment"] == category).astype(int)
    return df

## 9. Pickup / Delivery Encoding - Smoothed Target Encoding with Unseen-City Fallback

`pickup` and `delivery` have 64 categories each in training, too many to one-hot cleanly, and - critically - **8 cities appear in `validation.csv` that never appear in training** (Section 15 of the EDA). The encoder below addresses both:

- **Smoothing towards the global mean rate**, weighted by how many times each city appears, so low-volume cities don't get noisy, overconfident encodings (`smoothing` controls how many "virtual" global-mean observations are blended in - higher values shrink low-volume cities harder toward the global mean).
- **Explicit unseen-category fallback**: any city not present in the fitted mapping is encoded as the global mean, rather than raising an error or becoming `NaN`.
- **Out-of-fold (OOF) fitting for the training set itself**, so the encoding used to train a model on a given row was never computed *using that row's own target value* - otherwise the encoding would leak the target and understate holdout/validation error.

In [9]:
def fit_target_encoder(df: pd.DataFrame, column: str, target_column: str,
                        global_mean: float, smoothing: float = 20) -> pd.Series:
    """Returns a category -> smoothed mean-target mapping fit on df."""
    stats = df.groupby(column)[target_column].agg(["mean", "count"])
    smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
    return smoothed


def apply_target_encoder(df: pd.DataFrame, column: str, mapping: pd.Series, global_mean: float) -> pd.Series:
    """Unseen categories fall back to global_mean instead of becoming NaN or raising."""
    return df[column].map(mapping).fillna(global_mean)


def fit_target_encoder_oof(df: pd.DataFrame, column: str, target_column: str, global_mean: float,
                            smoothing: float = 20, n_splits: int = 5,
                            random_state: int = RANDOM_STATE) -> pd.Series:
    """Leak-free encoding for the TRAINING set: each row is encoded using a mapping fit only on the
    other folds, so a row's own target never influences its own encoded value."""
    oof_values = pd.Series(index=df.index, dtype=float)
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for fit_idx, holdout_idx in kfold.split(df):
        fold_mapping = fit_target_encoder(df.iloc[fit_idx], column, target_column, global_mean, smoothing)
        oof_values.iloc[holdout_idx] = df.iloc[holdout_idx][column].map(fold_mapping).values
    return oof_values.fillna(global_mean)

In [10]:
# Quick sanity check: unseen-city fallback behaves as expected
_global_mean_check = train_raw["posted_rate"].mean()
_pickup_map_check = fit_target_encoder(clean_weight(train_raw), "pickup", "posted_rate", _global_mean_check)

unseen_cities = sorted(set(validation_raw["pickup"]) - set(train_raw["pickup"]))
print("Unseen validation pickup cities:", unseen_cities)
print("Global mean posted_rate:", round(_global_mean_check, 2))
print("Encoded value assigned to unseen cities:",
      apply_target_encoder(validation_raw[validation_raw["pickup"].isin(unseen_cities)],
                            "pickup", _pickup_map_check, _global_mean_check).unique())

Unseen validation pickup cities: ['Allentown', 'Charlotte', 'Chicago', 'Jackson', 'Knoxville', 'Laredo', 'Norfolk', 'San Diego']
Global mean posted_rate: 2373.98
Encoded value assigned to unseen cities: [2373.98068229]


## 10. Time-Based Train / Holdout Split

Because the real prediction targets (`validation.csv` = November, December chart = December) both sit **after** the training window (Jan-Oct), a random split of `train-test.csv` would let the model "peek" at rows from the same weeks it's tested on and overstate accuracy. Instead, the **last 6 weeks of training data (Sep 19 - Oct 31) are held out by date** to mimic the real forecasting gap, leaving roughly an 86/14 split.

In [11]:
HOLDOUT_WEEKS = 6

split_cutoff = train_raw["date"].max() - pd.Timedelta(weeks=HOLDOUT_WEEKS)

dev_raw = train_raw[train_raw["date"] <= split_cutoff].reset_index(drop=True)
holdout_raw = train_raw[train_raw["date"] > split_cutoff].reset_index(drop=True)

print(f"Split cutoff: {split_cutoff.date()}")
print(f"Dev:     {len(dev_raw):,} rows  ({dev_raw['date'].min().date()} to {dev_raw['date'].max().date()})")
print(f"Holdout: {len(holdout_raw):,} rows  ({holdout_raw['date'].min().date()} to {holdout_raw['date'].max().date()})")
print(f"Dev share: {len(dev_raw) / len(train_raw) * 100:.1f}%")

Split cutoff: 2025-09-19
Dev:     41,468 rows  (2025-01-01 to 2025-09-19)
Holdout: 6,532 rows  (2025-09-20 to 2025-10-31)
Dev share: 86.4%


## 11. Assemble the Full Feature-Building Pipeline

All the pieces above are combined into one `build_features` function. It's parameterized by externally-fitted statistics (imputation values, encoder mappings) so the *exact same* fitted values used on training data are reused - never refit - on validation and December, which is what makes the pipeline leak-free and reproducible.

`has_market_signals` controls whether the market-signal columns and their interaction terms are included, so the same function produces both the full feature schema (train / validation) and the reduced schema required for `december-chart-inputs.csv`.

In [12]:
def build_features(df: pd.DataFrame, *, reference_date: pd.Timestamp, fill_values: dict,
                    pickup_map: pd.Series, delivery_map: pd.Series, global_mean_rate: float,
                    pickup_encoded: pd.Series = None, delivery_encoded: pd.Series = None,
                    has_market_signals: bool = True) -> pd.DataFrame:
    """
    Builds the model-ready feature table for one dataset.

    pickup_encoded / delivery_encoded: pass pre-computed OOF encodings here when building the
    TRAINING feature table (see Section 12). Leave as None to encode via the fitted mapping
    (used for holdout / validation / December, and for the training set's own production copy).
    """
    df = clean_weight(df)
    df = add_missing_indicators(df)
    df = impute_numeric(df, fill_values)
    df = add_distance_features(df)
    df = add_temporal_features(df, reference_date)
    if has_market_signals:
        df = add_market_interactions(df)
    df = one_hot_equipment(df)

    df["pickup_te"] = (
        pickup_encoded.values if pickup_encoded is not None
        else apply_target_encoder(df, "pickup", pickup_map, global_mean_rate)
    )
    df["delivery_te"] = (
        delivery_encoded.values if delivery_encoded is not None
        else apply_target_encoder(df, "delivery", delivery_map, global_mean_rate)
    )
    return df

## 12. Fit Encoders on `dev` Only → Transform `dev` and `holdout`

This produces the feature tables used for **internal model selection** in the modeling notebook. Fitting the imputation values and target encoders on `dev` alone (never on `holdout`) keeps the holdout metric honest - it's meant to simulate genuinely unseen future data, the same way `validation.csv` will be.

In [13]:
dev_clean = clean_weight(dev_raw)

fill_values_dev = {
    "weight": dev_clean["weight"].median(),
    "market_index": dev_clean["market_index"].median(),
}
global_mean_rate_dev = dev_clean["posted_rate"].mean()

pickup_map_dev = fit_target_encoder(dev_clean, "pickup", "posted_rate", global_mean_rate_dev)
delivery_map_dev = fit_target_encoder(dev_clean, "delivery", "posted_rate", global_mean_rate_dev)

# Out-of-fold encoding for dev itself (leak-free); holdout uses the plain fitted mapping since
# none of its rows were used to fit that mapping in the first place.
pickup_oof_dev = fit_target_encoder_oof(dev_clean, "pickup", "posted_rate", global_mean_rate_dev)
delivery_oof_dev = fit_target_encoder_oof(dev_clean, "delivery", "posted_rate", global_mean_rate_dev)

dev_features = flag_rpm_outliers(build_features(
    dev_raw, reference_date=REFERENCE_DATE, fill_values=fill_values_dev,
    pickup_map=pickup_map_dev, delivery_map=delivery_map_dev, global_mean_rate=global_mean_rate_dev,
    pickup_encoded=pickup_oof_dev, delivery_encoded=delivery_oof_dev,
))

holdout_features = flag_rpm_outliers(build_features(
    holdout_raw, reference_date=REFERENCE_DATE, fill_values=fill_values_dev,
    pickup_map=pickup_map_dev, delivery_map=delivery_map_dev, global_mean_rate=global_mean_rate_dev,
))

print("dev_features:", dev_features.shape)
print("holdout_features:", holdout_features.shape)

dev_features: (41468, 30)
holdout_features: (6532, 30)


## 13. Fit "Production" Encoders on the Full Training Set

Once model selection is done using the dev/holdout split above, the **final** model that generates `validation_predictions.csv` and the December chart should be trained on all available labeled data. So a second set of encoders is fit on the **full** `train-test.csv` and applied to the full training set, `validation.csv`, and `december-chart-inputs.csv` - this is the feature set the modeling notebook will use to produce the actual submission.

In [14]:
train_clean = clean_weight(train_raw)

fill_values_full = {
    "weight": train_clean["weight"].median(),
    "market_index": train_clean["market_index"].median(),
}
global_mean_rate_full = train_clean["posted_rate"].mean()

pickup_map_full = fit_target_encoder(train_clean, "pickup", "posted_rate", global_mean_rate_full)
delivery_map_full = fit_target_encoder(train_clean, "delivery", "posted_rate", global_mean_rate_full)

pickup_oof_full = fit_target_encoder_oof(train_clean, "pickup", "posted_rate", global_mean_rate_full)
delivery_oof_full = fit_target_encoder_oof(train_clean, "delivery", "posted_rate", global_mean_rate_full)

train_features_full = flag_rpm_outliers(build_features(
    train_raw, reference_date=REFERENCE_DATE, fill_values=fill_values_full,
    pickup_map=pickup_map_full, delivery_map=delivery_map_full, global_mean_rate=global_mean_rate_full,
    pickup_encoded=pickup_oof_full, delivery_encoded=delivery_oof_full,
))

validation_features = build_features(
    validation_raw, reference_date=REFERENCE_DATE, fill_values=fill_values_full,
    pickup_map=pickup_map_full, delivery_map=delivery_map_full, global_mean_rate=global_mean_rate_full,
)

print("train_features_full:", train_features_full.shape)
print("validation_features:", validation_features.shape)

# confirm the unseen validation cities still resolve safely with the production mapping
print("Any NaN in validation pickup_te / delivery_te?",
      validation_features[["pickup_te", "delivery_te"]].isna().any().any())

train_features_full: (48000, 30)
validation_features: (12000, 28)
Any NaN in validation pickup_te / delivery_te? False


## 14. Reduced Feature Schema for December (No Market Signals)

`december-chart-inputs.csv` never had `market_index`/`quote_signal` to begin with, so `build_features(..., has_market_signals=False)` naturally skips those columns and their interaction terms. A matching **reduced-schema copy of the full training set** is also produced here (same rows, market-signal columns dropped) - this is what the modeling notebook will use to train the fallback model that actually generates the December predictions, so the fallback model's training features match December's available features exactly.

In [15]:
december_features = build_features(
    december_raw, reference_date=REFERENCE_DATE, fill_values=fill_values_full,
    pickup_map=pickup_map_full, delivery_map=delivery_map_full, global_mean_rate=global_mean_rate_full,
    has_market_signals=False,
)

market_cols = ["market_index", "quote_signal", "market_index_missing",
               "market_index_x_distance", "quote_signal_x_distance"]
train_features_no_market = train_features_full.drop(columns=[c for c in market_cols if c in train_features_full.columns])

print("december_features:", december_features.shape)
print("train_features_no_market:", train_features_no_market.shape)
print()
print("december_features columns:", december_features.columns.tolist())

december_features: (31, 19)
train_features_no_market: (48000, 25)

december_features columns: ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate', 'weight_missing', 'log_distance', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'days_since_start', 'equip_Dry_Van', 'equip_Reefer', 'equip_Flatbed', 'pickup_te', 'delivery_te']


## 15. Final Feature Set - Overview

In [16]:
model_feature_cols_full = [
    "distance", "log_distance", "weight", "weight_missing",
    "market_index", "market_index_missing", "quote_signal",
    "market_index_x_distance", "quote_signal_x_distance",
    "month_sin", "month_cos", "dow_sin", "dow_cos", "days_since_start",
    "equip_Dry_Van", "equip_Reefer", "equip_Flatbed",
    "pickup_te", "delivery_te",
]

model_feature_cols_no_market = [c for c in model_feature_cols_full if c not in
                                 ["market_index", "market_index_missing", "quote_signal",
                                  "market_index_x_distance", "quote_signal_x_distance"]]

print(f"Full feature set  ({len(model_feature_cols_full)} features):")
print(model_feature_cols_full)
print()
print(f"No-market feature set ({len(model_feature_cols_no_market)} features):")
print(model_feature_cols_no_market)
print()
train_features_full[model_feature_cols_full + ["posted_rate"]].describe().T

Full feature set  (19 features):
['distance', 'log_distance', 'weight', 'weight_missing', 'market_index', 'market_index_missing', 'quote_signal', 'market_index_x_distance', 'quote_signal_x_distance', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'days_since_start', 'equip_Dry_Van', 'equip_Reefer', 'equip_Flatbed', 'pickup_te', 'delivery_te']

No-market feature set (14 features):
['distance', 'log_distance', 'weight', 'weight_missing', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'days_since_start', 'equip_Dry_Van', 'equip_Reefer', 'equip_Flatbed', 'pickup_te', 'delivery_te']



,count,mean,std,min,25%,50%,75%,max
distance,48000.0,1135.856654,728.564416,70.000000,550.400000,953.300000,1645.525000,3439.800000
log_distance,48000.0,6.799190,0.739436,4.262680,6.312460,6.860978,7.406422,8.143459
weight,48000.0,31417.741438,7971.489014,5000.000000,25961.000000,31496.000000,37018.000000,47500.000000
weight_missing,48000.0,0.006250,0.078810,0.000000,0.000000,0.000000,0.000000,1.000000
market_index,48000.0,1.083172,0.167452,0.676390,0.950610,1.055800,1.218525,1.467780
market_index_missing,48000.0,0.007792,0.087927,0.000000,0.000000,0.000000,0.000000,1.000000
quote_signal,48000.0,2.062468,0.291391,0.692280,1.891030,2.055750,2.221685,3.610350
market_index_x_distance,48000.0,1230.056666,819.725682,54.784800,586.605339,1021.537806,1736.982403,4615.563702
quote_signal_x_distance,48000.0,2330.515248,1503.573255,52.187100,1151.468290,1963.217077,3326.169166,8138.262460
month_sin,48000.0,0.102883,0.728926,-0.999999,-0.691351,0.265563,0.800980,0.999986


## 16. Save Processed Datasets & Encoder Artifacts

Everything downstream (`03_modeling.ipynb` and the final scoring/prediction script) reads from `data/processed/` rather than repeating this logic, so the feature engineering is defined in exactly one place.

- `*_features.csv` - model-ready feature tables.
- `feature_config.pkl` - the fitted imputation values, target-encoding mappings, global means, reference date, and column lists, so any future dataset (or a re-run of December/validation) can be transformed identically without re-fitting.

In [17]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Internal (dev/holdout) tables — for model selection & honest error estimates
dev_features.to_csv(processed_dir / "dev_features.csv", index=False)
holdout_features.to_csv(processed_dir / "holdout_features.csv", index=False)

# Production tables — for training the final submitted model(s)
train_features_full.to_csv(processed_dir / "train_features_full.csv", index=False)
train_features_no_market.to_csv(processed_dir / "train_features_no_market.csv", index=False)
validation_features.to_csv(processed_dir / "validation_features.csv", index=False)
december_features.to_csv(processed_dir / "december_features.csv", index=False)

feature_config = {
    "reference_date": REFERENCE_DATE,
    "random_state": RANDOM_STATE,
    "rpm_bounds": (RPM_LOWER, RPM_UPPER),
    "model_feature_cols_full": model_feature_cols_full,
    "model_feature_cols_no_market": model_feature_cols_no_market,
    # dev-fit artifacts (internal validation)
    "fill_values_dev": fill_values_dev,
    "global_mean_rate_dev": global_mean_rate_dev,
    "pickup_map_dev": pickup_map_dev,
    "delivery_map_dev": delivery_map_dev,
    # full-train-fit artifacts (production / submission)
    "fill_values_full": fill_values_full,
    "global_mean_rate_full": global_mean_rate_full,
    "pickup_map_full": pickup_map_full,
    "delivery_map_full": delivery_map_full,
}

with open(processed_dir / "feature_config.pkl", "wb") as f:
    pickle.dump(feature_config, f)

print("Saved processed datasets and feature_config.pkl to", processed_dir.resolve())
for p in sorted(processed_dir.iterdir()):
    print(" -", p.name)

Saved processed datasets and feature_config.pkl to C:\Users\amc\Desktop\freight-rate-prediction\data\processed
 - december_features.csv
 - dev_features.csv
 - feature_config.pkl
 - holdout_features.csv
 - train_features_full.csv
 - train_features_no_market.csv
 - validation_features.csv


## 17. Summary

- **Cleaning**: `abs(weight)` corrects the sign-flip error; `weight`/`market_index` missingness is flagged with an indicator, then median-imputed.
- **Distance**: raw + `log1p(distance)` to help linear models capture the diminishing rate-per-mile curve.
- **Temporal**: cyclical (`sin`/`cos`) month and day-of-week encodings extrapolate sensibly into November/December; `days_since_start` captures any residual linear trend.
- **Market signals**: `market_index`/`quote_signal` interacted with `distance`, matching the multiplicative relationship found in the EDA. Never used for December, which lacks both columns entirely.
- **Equipment**: plain one-hot (low cardinality, fully shared across all files).
- **Pickup/Delivery**: smoothed, out-of-fold target encoding with an explicit global-mean fallback for the 8 cities seen in `validation.csv` but never in training.
- **Split**: time-based (last 6 weeks of training as holdout), not random - the model must be evaluated the way it will actually be used: forecasting forward past the end of its training window.
- **Two feature schemas** are produced and saved: the full schema (train/holdout/validation, includes market signals) and a reduced schema (train-no-market/December, excludes them) so a dedicated fallback model can be trained on directly comparable features to what December provides.

Next: `03_modeling.ipynb` trains and compares candidate models on `dev_features.csv` / `holdout_features.csv`, selects a final model (and a no-market fallback variant), then generates `validation_predictions.csv` and the December predictions for `score.py`.